# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [ ]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\n  OHLCV: {DATA_PATH_OHLCV}\n  Indices: {DATA_PATH_INDICES}")

## II. Data Pipeline

In [ ]:
# === RELOADING FROM PARQUET ===
features_df = pd.read_parquet("output/features_df.parquet")
macro_df = pd.read_parquet("output/macro_df.parquet")
df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

In [ ]:
print(f"features_df:\n{features_df}\n")

In [ ]:
# print("⏳ Loading DataFrames... takes ~6 minutes")
# df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
# df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
# df_fed = pd.read_parquet(
#     OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
# )

# print("⚡ Generating Features...")
# features_df, macro_df = generate_features(
#     df_ohlcv=df_ohlcv,
#     df_indices=df_indices,
#     df_fed=df_fed,
#     benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
# )

# print("🚀 Unstacking Wide Matrices...")

# df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
# df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
# df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
#     df_close_wide = df_close_wide.replace(0, np.nan)

# df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
# df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

# print(
#     f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
# )

In [ ]:
# # === PERSISTENCE SANDBOX ===
# features_df.to_parquet("output/features_df.parquet", index=True)
# macro_df.to_parquet("output/macro_df.parquet", index=True)
# df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
# df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
# df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

## III. The Analysis Suite

In [ ]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

In [ ]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=10,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

In [ ]:
##################################
##################################
##################################

In [ ]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

In [ ]:
# Set the start_date index here
start_date_idx = 42

# Unpack the next 4 consecutive values
start_date, decision_date, buy_date, holding_end_date = [
    pd.Timestamp(SU.peek(i, result)) for i in range(start_date_idx, start_date_idx + 4)
]

# Print to verify
print(f"start_date:        {start_date.date()}")
print(f"decision_date:     {decision_date.date()}")
print(f"buy_date:          {buy_date.date()}")
print(f"holding_end_date:  {holding_end_date.date()}")

In [ ]:
target_tickers = SU.peek(4, result)

In [ ]:
# Ensure IndexSlice is defined for MultiIndex slicing
idx = pd.IndexSlice

# 1. Define the Date Ranges for each period
# (Start, End) tuples for each slice
period_definitions = {
    "Full Period": (start_date, holding_end_date),
    "Lookback": (start_date, decision_date),
    "Holding": (buy_date, holding_end_date),
}

# Dictionary to store results
portfolio_results = {}

print("--- Portfolio Performance Calculation ---")

for period_name, (s_date, e_date) in period_definitions.items():
    # A. Slice: Extract [Ticker, Date] range for Adj Close
    df_slice = df_ohlcv.loc[idx[target_tickers, s_date:e_date], ["Adj Close"]]

    if df_slice.empty:
        print(f"Warning: No data found for {period_name} ({s_date} to {e_date})")
        continue

    # B. Normalize: Calculate price relative to the first available price in this specific slice
    # transform('first') ensures we divide every ticker's price by its own starting price
    first_prices = df_slice.groupby(level=0)["Adj Close"].transform("first")
    df_norm = (df_slice["Adj Close"] / first_prices).to_frame(name="Normalized Close")

    # C. Final Return: Get the last normalized price for each ticker
    # This represents (Ending Price / Starting Price) for each asset
    final_norm_prices = df_norm.groupby(level=0).last()

    # D. Portfolio Log Gain:
    # 1. Mean() calculates the arithmetic return of an equally weighted portfolio
    # 2. np.log() converts the total arithmetic return into log space
    log_gain = np.log(final_norm_prices.mean()).iloc[0]

    # Store result
    portfolio_results[period_name] = log_gain

    print(
        f"{period_name:12} | Log Gain: {log_gain:.4f} | Range: {s_date.date()} to {e_date.date()}"
    )

# Accessing specific values later:
# lookback_val = portfolio_results["Lookback"]

In [ ]:
# Ensure IndexSlice is defined
idx = pd.IndexSlice

# 1. Define the Date Ranges (matching cell 15)
period_definitions = {
    "Full Period": (start_date, holding_end_date),
    "Lookback": (start_date, decision_date),
    "Holding": (buy_date, holding_end_date),
}

portfolio_sharpe_results = {}

print("--- Portfolio Sharpe Calculation (Annualized) ---")

for period_name, (s_date, e_date) in period_definitions.items():
    # A. Slice: Extract [Ticker, Date] range
    df_slice = df_ohlcv.loc[idx[target_tickers, s_date:e_date], ["Adj Close"]]

    if df_slice.empty:
        continue

    # B. Pivot to Wide: [Date x Ticker]
    df_wide = df_slice["Adj Close"].unstack(level=0)

    # C. Equity Curve: Calculate arithmetic Buy & Hold equity
    # We normalize each ticker by its first available price in this slice
    df_norm = df_wide / df_wide.bfill().iloc[0]
    equity_curve = df_norm.mean(axis=1)

    # D. Portfolio Returns: Daily percentage change of the equity curve
    # Matches PerformanceCalculator / QuantUtils.compute_portfolio_stats
    portf_rets = equity_curve.pct_change().dropna()

    # E. Calculate Sharpe: Independent Manual Calculation
    mu = portf_rets.mean()
    sigma = portf_rets.std()
    annual_period = GLOBAL_SETTINGS.get("annual_period", 252)
    sharpe = (mu / sigma) * np.sqrt(annual_period) if sigma > 0 else 0.0

    portfolio_sharpe_results[period_name] = sharpe

    print(
        f"{period_name:12} | Sharpe: {sharpe:7.4f} | (μ: {mu:8.6f}, σ: {sigma:8.6f}) | Range: {s_date.date()} to {e_date.date()}"
    )

In [ ]:
# Ensure IndexSlice is defined
idx = pd.IndexSlice

# 1. Define the Date Ranges
period_definitions = {
    "Full Period": (start_date, holding_end_date),
    "Lookback": (start_date, decision_date),
    "Holding": (buy_date, holding_end_date),
}

portfolio_sharpe_atrp_results = {}

print("--- Portfolio Sharpe (ATRP) Calculation ---")

for period_name, (s_date, e_date) in period_definitions.items():
    # A. Slice: Extract Prices from df_ohlcv and ATRP from features_df
    try:
        # Get Prices and Pivot to Wide Format
        df_prices = df_ohlcv.loc[
            idx[target_tickers, s_date:e_date], ["Adj Close"]
        ].unstack(level=0)["Adj Close"]

        # Get ATRP from features_df (where it actually exists) and Pivot to Wide Format
        df_atrp_slice = features_df.loc[
            idx[target_tickers, s_date:e_date], ["ATRP"]
        ].unstack(level=0)["ATRP"]
    except KeyError as e:
        print(f"Skipping {period_name}: Missing data in range ({e})")
        continue

    if df_prices.empty or df_atrp_slice.empty:
        continue

    # B. Equity Curve & Portfolio Returns
    # Normalize prices to the start of the specific period
    norm_prices = df_prices / df_prices.bfill().iloc[0]
    # Simple average for an equally weighted portfolio equity curve
    equity_curve = norm_prices.mean(axis=1)
    portf_rets = equity_curve.pct_change()

    # C. Price-Drift Weighted Volatility (ATRP)
    # Weights drift based on performance: weight_i(t) = P_i(t) / sum(Prices(t))
    current_weights = norm_prices.div(norm_prices.sum(axis=1), axis=0)

    # Dot product of weights and ATRP for daily portfolio-level volatility
    portf_atrp = (current_weights * df_atrp_slice).sum(axis=1)

    # D. Sharpe (ATRP) Formula
    # Align: Calculate means only for rows where returns exist (skips the first NaN return)
    mask = portf_rets.notna()
    avg_ret = portf_rets.where(mask).mean()
    avg_vol = portf_atrp.where(mask).mean()

    # Avoid division by zero
    sharpe_atrp = avg_ret / avg_vol if avg_vol > 0 else 0.0

    portfolio_sharpe_atrp_results[period_name] = sharpe_atrp

    print(
        f"{period_name:12} | Sharpe(ATRP): {sharpe_atrp:7.4f} | "
        f"(μ_ret: {avg_ret:8.6f}, μ_atrp: {avg_vol:8.6f}) | "
        f"Range: {s_date.date()} to {e_date.date()}"
    )

In [ ]:
# Ensure IndexSlice is defined
idx = pd.IndexSlice

# 1. Define the Date Ranges (Consistent with previous calculations)
period_definitions = {
    "Full Period": (start_date, holding_end_date),
    "Lookback": (start_date, decision_date),
    "Holding": (buy_date, holding_end_date),
}

portfolio_sharpe_trp_results = {}

print("--- Portfolio Sharpe (TRP) Calculation ---")

for period_name, (s_date, e_date) in period_definitions.items():
    try:
        # A. Slice: Prices from df_ohlcv and TRP from features_df
        df_prices = df_ohlcv.loc[
            idx[target_tickers, s_date:e_date], ["Adj Close"]
        ].unstack(level=0)["Adj Close"]

        # Use 'TRP' column from features_df
        df_trp_slice = features_df.loc[
            idx[target_tickers, s_date:e_date], ["TRP"]
        ].unstack(level=0)["TRP"]

    except KeyError as e:
        print(f"Skipping {period_name}: Missing data in range ({e})")
        continue

    if df_prices.empty or df_trp_slice.empty:
        continue

    # B. Equity Curve & Portfolio Returns
    # Re-base to 1.0 at the start of each period
    norm_prices = df_prices / df_prices.bfill().iloc[0]
    equity_curve = norm_prices.mean(axis=1)
    portf_rets = equity_curve.pct_change()

    # C. Price-Drift Weighted Volatility (TRP)
    # Portfolio volatility must account for the fact that winning stocks grow to represent
    # a larger portion of the daily volatility.
    current_weights = norm_prices.div(norm_prices.sum(axis=1), axis=0)
    portf_trp = (current_weights * df_trp_slice).sum(axis=1)

    # D. Manual Sharpe (TRP) Formula
    mask = portf_rets.notna()
    avg_ret = portf_rets.where(mask).mean()
    avg_vol_trp = portf_trp.where(mask).mean()

    # Sharpe(TRP) = Average Daily Return / Average Daily True Range Percent
    sharpe_trp = avg_ret / avg_vol_trp if avg_vol_trp > 0 else 0.0

    portfolio_sharpe_trp_results[period_name] = sharpe_trp

    print(
        f"{period_name:12} | Sharpe(TRP) : {sharpe_trp:7.4f} | "
        f"(μ_ret: {avg_ret:8.6f}, μ_trp: {avg_vol_trp:8.6f}) | "
        f"Range: {s_date.date()} to {e_date.date()}"
    )

In [ ]:
##################################
##################################
##################################

In [ ]:
# 1. Setup Parameters
# NVDA had a dividend payment date on 2026-04-01
verification_ticker = "NVDA"
# Changed to the last date shown in your features_df.info()
target_date = pd.Timestamp("2026-04-24")
lookback_days = 120

# 2. Extract Data Safely
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")

# --- DATE VALIDATION ---
# If our chosen target_date is not in the index, grab the most recent available one
if target_date not in ticker_prices.index:
    new_target = ticker_prices.index[ticker_prices.index <= target_date][-1]
    print(
        f"Adjusting target date from {target_date.date()} to {new_target.date()} (Last available trading day)"
    )
    target_date = new_target

# Slice the previous 100+ days for rolling calculations
start_date_slice = target_date - pd.Timedelta(days=lookback_days)
raw_slice = ticker_prices.loc[:target_date].tail(90).copy()  # Get enough history
feat_row = ticker_features.loc[target_date]

# --- 3. Manual Calculations (The "Easy" Group) ---

# A. Ret_1d: (Current / Previous) - 1
manual_ret_1d = raw_slice["Adj Close"].pct_change().iloc[-1]

# B. Mom_21: (Current / Price 21-trading-bars ago) - 1
manual_mom_21 = raw_slice["Adj Close"].pct_change(21).iloc[-1]

# C. DD_21: (Current / Max in last 21 trading bars) - 1
rolling_max_21 = raw_slice["Adj Close"].rolling(window=21).max()
manual_dd_21 = (raw_slice["Adj Close"] / rolling_max_21 - 1).iloc[-1]

# D. Consistency: % of last 5 trading days where return was positive
daily_rets = raw_slice["Adj Close"].pct_change()
manual_consist = (daily_rets > 0).rolling(window=5).mean().iloc[-1]

# E. TRP (True Range Percent): Raw daily volatility
prev_close = raw_slice["Adj Close"].shift(1)
tr_val = pd.concat(
    [
        (raw_slice["Adj High"] - raw_slice["Adj Low"]),
        (raw_slice["Adj High"] - prev_close).abs(),
        (raw_slice["Adj Low"] - prev_close).abs(),
    ],
    axis=1,
).max(axis=1)
manual_trp = (tr_val / raw_slice["Adj Close"]).iloc[-1]

# 4. Comparison Table
easy_verification = pd.DataFrame(
    {
        "Feature": ["Ret_1d", "Mom_21", "DD_21", "Consistency", "TRP"],
        "Manual_Value": [
            manual_ret_1d,
            manual_mom_21,
            manual_dd_21,
            manual_consist,
            manual_trp,
        ],
        "Feature_DF_Value": [
            feat_row["Ret_1d"],
            feat_row["Mom_21"],
            feat_row["DD_21"],
            feat_row["Consistency"],
            feat_row["TRP"],
        ],
    }
)

easy_verification["Abs_Error"] = (
    easy_verification["Manual_Value"] - easy_verification["Feature_DF_Value"]
).abs()

print(f"\nVerification for {verification_ticker} on {target_date.date()}")
print(
    easy_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:.6f}".format,
            "Feature_DF_Value": "{:.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if easy_verification["Abs_Error"].max() < 1e-10:
    print("\n✅ Easy Group Verified.")
else:
    print("\n❌ Discrepancy detected.")

In [ ]:
# 1. Setup Data for NVDA (Using the full history for smoothing warm-up)
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

# For smoothing, we need a long history to match Wilder's logic exactly
adj_close = ticker_prices["Adj Close"]
adj_high = ticker_prices["Adj High"]
adj_low = ticker_prices["Adj Low"]
rsi_period = GLOBAL_SETTINGS["rsi_period"]  # usually 14
atr_period = GLOBAL_SETTINGS["atr_period"]  # usually 14

# --- 2. Manual Calculations (Group 2) ---

# A. RSI (Wilder's Smoothing)
delta = adj_close.diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

# Wilder's smoothing is an EWM with alpha = 1/period
avg_gain = gain.ewm(alpha=1 / rsi_period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1 / rsi_period, adjust=False).mean()

rs = avg_gain / avg_loss
manual_rsi = (100 - (100 / (1 + rs))).loc[target_date]

# B. ATRP (Average True Range Percent)
# First get True Range (TR)
prev_close = adj_close.shift(1)
tr = pd.concat(
    [(adj_high - adj_low), (adj_high - prev_close).abs(), (adj_low - prev_close).abs()],
    axis=1,
).max(axis=1)

# ATR is Wilder's smoothing of TR
manual_atr = tr.ewm(alpha=1 / atr_period, adjust=False).mean()
manual_atrp = (manual_atr / adj_close).loc[target_date]

# C. AutoCorr_15 (1-day lag correlation)
rets = adj_close.pct_change()
# Calculate rolling correlation of returns vs lagged returns
manual_autocorr = rets.rolling(window=15).corr(rets.shift(1)).loc[target_date]

# --- 3. Comparison Table ---
group2_verification = pd.DataFrame(
    {
        "Feature": ["RSI", "ATRP", "AutoCorr_15"],
        "Manual_Value": [manual_rsi, manual_atrp, manual_autocorr],
        "Feature_DF_Value": [
            feat_row["RSI"],
            feat_row["ATRP"],
            feat_row["AutoCorr_15"],
        ],
    }
)

group2_verification["Abs_Error"] = (
    group2_verification["Manual_Value"] - group2_verification["Feature_DF_Value"]
).abs()

print(f"Verification for {verification_ticker} on {target_date.date()}")
print(
    group2_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:.6f}".format,
            "Feature_DF_Value": "{:.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if group2_verification["Abs_Error"].max() < 1e-8:
    print("\n✅ Smoothed Group Verified.")
else:
    print(
        "\n❌ Discrepancy detected. This often happens if the 'QuantUtils' uses a different smoothing initialization than 'adjust=False'."
    )

In [ ]:
# 1. Setup Parameters from GLOBAL_SETTINGS
# We use the 252-day window as confirmed by our previous diagnostic
verification_ticker = "NVDA"
target_date = pd.Timestamp("2026-04-24")
q_window = GLOBAL_SETTINGS["quality_window"]  # 252 days
q_min = GLOBAL_SETTINGS["quality_min_periods"]  # 126 days

# 2. Extract Slices
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

# --- 3. MANUAL CALCULATIONS ---

# A. Data Health Metrics (Staleness & Frozen Volume)
# Logic: Identifies low-quality data or "zombie" tickers
is_stale = (
    (ticker_prices["Volume"] == 0)
    | (ticker_prices["Adj High"] == ticker_prices["Adj Low"])
).astype(int)
manual_stale_pct = (
    is_stale.rolling(window=q_window, min_periods=q_min).mean().loc[target_date]
)

has_same_vol = (ticker_prices["Volume"].diff() == 0).astype(int)
manual_same_vol = (
    has_same_vol.rolling(window=q_window, min_periods=q_min).sum().loc[target_date]
)

# B. Liquidity Metric (RollMedDollarVol)
# Logic: Measures typical daily dollar turnover over 1 year (252 days).
# Using Median instead of Mean protects the filter from being skewed by single-day volume spikes.
dollar_vol = ticker_prices["Adj Close"] * ticker_prices["Volume"]
manual_med_dvol = (
    dollar_vol.rolling(window=q_window, min_periods=q_min).median().loc[target_date]
)

# C. Trend Acceleration (Convexity)
# Logic: Calculated as the 2-day change in the 5-day price slope.
# A positive value indicates the trend is accelerating upward (Parabolic).
# A negative value indicates the trend is decelerating (Exhaustion).
s_idx = ticker_features.index.get_loc(target_date)
slope_t = ticker_features["Slope_P_5"].iloc[s_idx]
slope_t_minus_2 = ticker_features["Slope_P_5"].iloc[s_idx - 2]
manual_convexity = slope_t - slope_t_minus_2


# --- 4. VERIFICATION OUTPUT ---

final_verification = pd.DataFrame(
    {
        "Feature": [
            "RollingStalePct",
            "RollingSameVolCount",
            "RollMedDollarVol",
            "Convexity",
        ],
        "Manual_Value": [
            manual_stale_pct,
            manual_same_vol,
            manual_med_dvol,
            manual_convexity,
        ],
        "Feature_DF_Value": [
            feat_row["RollingStalePct"],
            feat_row["RollingSameVolCount"],
            feat_row["RollMedDollarVol"],
            feat_row["Convexity"],
        ],
    }
)

# Calculate error and formatting
final_verification["Abs_Error"] = (
    final_verification["Manual_Value"] - final_verification["Feature_DF_Value"]
).abs()

print(f"--- Final Group Verification: {verification_ticker} ({target_date.date()}) ---")
print(
    final_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:,.4f}".format,
            "Feature_DF_Value": "{:,.4f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

# Assertion Check
if final_verification["Abs_Error"].max() < 1e-7:
    print("\n✅ Final Group Verified: All quality and acceleration metrics match.")
else:
    print("\n❌ Discrepancy detected in final group.")

In [ ]:
# 1. Setup Data for Audit
verification_ticker = "NVDA"
target_date = pd.Timestamp("2026-04-24")
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

# Window constants from GLOBAL_SETTINGS
atr_period = GLOBAL_SETTINGS["atr_period"]  # 14
win_63 = GLOBAL_SETTINGS["63d_window"]  # 63
win_20 = 20  # Hardcoded in Range_Pos logic

# Slicing context
s_idx = ticker_prices.index.get_loc(target_date)
adj_close = ticker_prices["Adj Close"]
adj_high = ticker_prices["Adj High"]
adj_low = ticker_prices["Adj Low"]
volume = ticker_prices["Volume"]


# Helper: Weighted Slope Kernel confirmed in previous steps
def apply_5d_slope_kernel(series, idx):
    y = series.iloc[idx - 4 : idx + 1].values
    weights = np.array([-2, -1, 0, 1, 2])
    return np.sum(y * weights) / 10


# --- 2. MANUAL CALCULATIONS ---

# A. ATR (Average True Range)
# Logic: Wilder's smoothing of the True Range.
# Foundational for risk-adjusted position sizing.
prev_close = adj_close.shift(1)
tr = pd.concat(
    [(adj_high - adj_low), (adj_high - prev_close).abs(), (adj_low - prev_close).abs()],
    axis=1,
).max(axis=1)
manual_atr = tr.ewm(alpha=1 / atr_period, adjust=False).mean().loc[target_date]

# B. Range_Pos_20 (Price Channel Position)
# Logic: Where the current price sits relative to the 20-day High/Low.
# 1.0 = at 20-day high, 0.0 = at 20-day low.
h_20 = adj_high.rolling(window=win_20).max().loc[target_date]
l_20 = adj_low.rolling(window=win_20).min().loc[target_date]
manual_range_pos = (adj_close.loc[target_date] - l_20) / (h_20 - l_20)

# C. Beta_63 & IR_63 (Relative Market Performance)
# Logic: Requires market returns from macro_df aligned with ticker returns.
stock_rets = adj_close.pct_change()
mkt_rets = macro_df["Mkt_Ret"].loc[stock_rets.index]

y = stock_rets.iloc[s_idx - 62 : s_idx + 1]
x_mkt = mkt_rets.iloc[s_idx - 62 : s_idx + 1]

# Beta: Cov(s, m) / Var(m)
manual_beta_63 = np.cov(y, x_mkt)[0, 1] / np.var(x_mkt, ddof=1)
# IR: Average Excess Return / Std of Excess Return
excess = y - x_mkt
manual_ir_63 = excess.mean() / excess_rets.std()  # Note: verified in previous run

# D. Slope_P_5 (Price Trend Velocity)
# Logic: 5-day OLS slope of Log Prices (continuous growth rate).
manual_slope_p = apply_5d_slope_kernel(np.log(adj_close), s_idx)

# E. Slope_V_5 (Volume Momentum)
# Logic: Slope of OBV after normalizing volume by a 63-day baseline.
v_baseline = volume.rolling(window=63).mean()
v_rel = volume / v_baseline
obv_rel = (np.sign(adj_close.diff()).fillna(0) * v_rel).cumsum()
manual_slope_v = apply_5d_slope_kernel(obv_rel, s_idx)

# --- 3. AUDIT SUMMARY TABLE ---

audit_results = pd.DataFrame(
    {
        "Feature": [
            "ATR",
            "IR_63",
            "Beta_63",
            "Range_Pos_20",
            "Slope_P_5",
            "Slope_V_5",
        ],
        "Manual_Value": [
            manual_atr,
            manual_ir_63,
            manual_beta_63,
            manual_range_pos,
            manual_slope_p,
            manual_slope_v,
        ],
        "Feature_DF_Value": [
            feat_row["ATR"],
            feat_row["IR_63"],
            feat_row["Beta_63"],
            feat_row["Range_Pos_20"],
            feat_row["Slope_P_5"],
            feat_row["Slope_V_5"],
        ],
    }
)

audit_results["Abs_Error"] = (
    audit_results["Manual_Value"] - audit_results["Feature_DF_Value"]
).abs()

print(f"--- Completion Audit: {verification_ticker} ({target_date.date()}) ---")
print(
    audit_results.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:,.6f}".format,
            "Feature_DF_Value": "{:,.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if audit_results["Abs_Error"].max() < 1e-7:
    print("\n✅ All remaining columns verified. Audit Complete.")
else:
    print("\n❌ Audit mismatch detected in remaining columns.")

In [ ]:
#########################
#########################
#########################

In [ ]:
# 1. Setup Audit for the whole Universe on one specific day
target_date = pd.Timestamp("2026-04-24")
verification_ticker = "NVDA"

# 2. Get the "Cross-Section" (All tickers for this date)
# We pull the raw convexity values for every ticker from features_df
universe_convexity = features_df.xs(target_date, level="Date")["Convexity"]

# 3. Manual "Engine" Logic (Replicating get_agent_view)
# Step A: Clean (Remove Inf/NaN)
clean_universe = universe_convexity.replace([np.inf, -np.inf], np.nan).dropna()

# Step B: Manual Z-Score
u_mean = clean_universe.mean()
u_std = clean_universe.std()
manual_z_scores = (clean_universe - u_mean) / u_std

# Step C: Neutralize & Clip (Filling NaNs with 0 as per .fillna(0).clip())
clip_val = GLOBAL_SETTINGS.get("feature_zscore_clip", 4.0)
final_agent_values = manual_z_scores.fillna(0).clip(-clip_val, clip_val)

# 4. Compare for NVDA
raw_val = clean_universe.loc[verification_ticker]
agent_val = final_agent_values.loc[verification_ticker]

print(f"--- Pillar 6 (Convexity) Cross-Sectional Audit ---")
print(f"Date: {target_date.date()} | Universe Size: {len(clean_universe)}")
print(f"Universe Mean Convexity: {u_mean:.6f}")
print(f"Universe Std Convexity : {u_std:.6f}")
print("-" * 50)
print(f"NVDA Raw Convexity    : {raw_val:.6f}")
print(f"NVDA Agent View (Z)   : {agent_val:.6f}")

# verification against the registry output (if you have the registry object 'pillar_6')
# pillar_6_view = pillar_6.get_agent_view(obs_for_that_day)

In [ ]:
from core.contracts import MetricBlueprint
from types import SimpleNamespace

# 1. Prepare the "obs" object that the Blueprint expects
# We take the features for ALL tickers on our target date
daily_snapshot = features_df.xs(target_date, level="Date")

# The Blueprint formula expects attributes like 'obs.convexity'
# We create a SimpleNamespace where columns are attributes
obs = SimpleNamespace(
    convexity=daily_snapshot["Convexity"],
    slope_p_5=daily_snapshot["Slope_P_5"],
    slope_v_5=daily_snapshot["Slope_V_5"],
)

# 2. Re-create the Blueprint exactly as it is in your registry.py
convexity_blueprint = MetricBlueprint(
    name="Convexity",
    category="Physics",
    regime="Acceleration",
    description="Second derivative of price.",
    agent_hint="The 'Golden Exit'.",
    intervention_trigger="EXIT LONG if Value < -0.7",
    scaling_type="Z-Score",
    formula=lambda obs: obs.convexity,
)

# 3. GET THE SYSTEM OUTPUT
# This runs the actual Engine logic: Clean -> Scale -> Clip
system_output_series = convexity_blueprint.get_agent_view(obs)
system_val = system_output_series.loc[verification_ticker]

# 4. FINAL VERIFICATION COMPARISON
print(f"--- Final System Verification: Pillar 6 ---")
print(f"Manual Audit Value (Z): {agent_val:.5f}")
print(f"System Output Value (Z): {system_val:.5f}")

error = abs(agent_val - system_val)
if error < 1e-5:
    print(
        f"\n✅ VERIFIED: The System Logic matches the Manual Audit (Error: {error:.2e})"
    )
else:
    print(f"\n❌ DISCREPANCY: System output differs from manual calculation!")

In [ ]:
import pandas as pd
import numpy as np
from types import SimpleNamespace


# --- 1. Mock QuantUtils to match the Registry's expected environment ---
class QuantUtils:
    @staticmethod
    def zscore(series: pd.Series) -> pd.Series:
        """Robust Z-score handling NaNs/Infs for cross-sectional data."""
        clean = series.replace([np.inf, -np.inf], np.nan)
        return (clean - clean.mean()) / clean.std()


# --- 2. Prepare the Observation (obs) for 2026-04-24 ---
target_date = pd.Timestamp("2026-04-24")
verification_ticker = "NVDA"
daily_snapshot = features_df.xs(target_date, level="Date")

obs = SimpleNamespace(
    slope_v_5=daily_snapshot["Slope_V_5"], slope_p_5=daily_snapshot["Slope_P_5"]
)

# --- 3. RE-CREATE BLUEPRINT (Exactly from registry.py) ---
divergence_blueprint = MetricBlueprint(
    name="OBV Divergence (5d)",
    category="Volume/Fuel",
    regime="Confirmation",
    description="Z-scored gap between relative volume flow and price trend.",
    agent_hint="Detects smart money accumulation/distribution.",
    intervention_trigger="INVALIDATE Longs if Divergence < -1.0std.",
    scaling_type="Z-Score",  # Triggers SECOND Z-score in get_agent_view
    formula=lambda obs: (
        QuantUtils.zscore(obs.slope_v_5) - QuantUtils.zscore(obs.slope_p_5)
    ),
)

# --- 4. MANUAL AUDIT (Step-by-Step Logic) ---

# Step A: Internal Formula Math (First Z-score)
z_vol = QuantUtils.zscore(obs.slope_v_5)
z_price = QuantUtils.zscore(obs.slope_p_5)
raw_divergence = z_vol - z_price

# Step B: Engine Scaling Math (Second Z-score)
# get_agent_view() cleans the raw output and Z-scores it AGAIN
final_manual_z = QuantUtils.zscore(raw_divergence)

# Step C: Neutralize & Clip
clip_val = GLOBAL_SETTINGS.get("feature_zscore_clip", 4.0)
manual_final_val = (
    final_manual_z.fillna(0).clip(-clip_val, clip_val).loc[verification_ticker]
)

# --- 5. SYSTEM COMPARISON ---
system_output_series = divergence_blueprint.get_agent_view(obs)
system_final_val = system_output_series.loc[verification_ticker]

# --- 6. RESULTS ---
print(f"--- Pillar 5 (OBV Divergence) Completion Audit ---")
print(f"NVDA Z-Volume (Internal): {z_vol.loc[verification_ticker]:.5f}")
print(f"NVDA Z-Price  (Internal): {z_price.loc[verification_ticker]:.5f}")
print(f"NVDA Raw Div (V_z - P_z): {raw_divergence.loc[verification_ticker]:.5f}")
print("-" * 50)
print(f"Manual Final Z-Score    : {manual_final_val:.5f}")
print(f"System Final Z-Score    : {system_final_val:.5f}")

error = abs(manual_final_val - system_final_val)
if error < 1e-5:
    print(
        f"\n✅ VERIFIED: Pillar 5 logic is mathematically consistent (Error: {error:.2e})"
    )
    print("The 'Double Z-Score' architecture is working as designed.")
else:
    print(f"\n❌ DISCREPANCY: Check QuantUtils.zscore implementation details.")